# 02 — Addiction Feature Analysis: Alcoholic vs Control

**Purpose:** Analyze which EEG features distinguish alcoholic from control subjects.
Key biomarkers: elevated beta power, reduced P300, altered alpha connectivity.

**Hypothesis:** Alcoholic subjects show hyperarousal (high beta), impaired attention (low P300),
and disrupted functional networks (altered connectivity).

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

from classifiers.addiction.data import load_uci_dataset, AddictionConfig, dataset_to_epochs
from classifiers.addiction.features import (
    AddictionFeatureConfig, compute_resting_beta,
    compute_p300_features, compute_frontal_beta_ratio,
)
from shared.features.spectral import compute_band_power, SpectralConfig

plt.style.use('dark_background')
UCI_DIR = Path('../../data/raw/uci_eeg')

## 1. Load Data and Create Epochs

In [ ]:
config = AddictionConfig(data_dir=UCI_DIR)
dataset = load_uci_dataset(config)
epochs = dataset_to_epochs(dataset, preprocess=True)

alc_idx = np.where(dataset.labels == 1)[0]
ctrl_idx = np.where(dataset.labels == 0)[0]
print(f'Alcoholic epochs: {len(alc_idx)}, Control epochs: {len(ctrl_idx)}')

## 2. Resting Beta Power — Hyperarousal Marker

Elevated beta (13-30 Hz) is a hallmark of addiction-related hyperarousal.

In [ ]:
feat_config = AddictionFeatureConfig()
beta = compute_resting_beta(epochs, feat_config)

beta_alc = beta[alc_idx].mean(axis=1)
beta_ctrl = beta[ctrl_idx].mean(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

bp = ax1.boxplot([beta_alc, beta_ctrl], labels=['Alcoholic', 'Control'],
                 patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#E57373')
bp['boxes'][1].set_facecolor('#4FC3F7')
for box in bp['boxes']:
    box.set_edgecolor('#333')
    box.set_linewidth(2)
t_stat, p_val = stats.ttest_ind(beta_alc, beta_ctrl)
ax1.set_ylabel('Beta Power')
ax1.set_title(f'Resting Beta: t={t_stat:.2f}, p={p_val:.4f}')

ax2.hist(beta_alc, bins=30, alpha=0.6, label='Alcoholic', color='#E57373', edgecolor='#333')
ax2.hist(beta_ctrl, bins=30, alpha=0.6, label='Control', color='#4FC3F7', edgecolor='#333')
ax2.set_xlabel('Beta Power')
ax2.legend()
ax2.set_title('Distribution')
plt.tight_layout()
plt.show()

## 3. P300 Amplitude — Attention Biomarker

Reduced P300 is one of the most replicated EEG findings in alcoholism research.

In [ ]:
p300 = compute_p300_features(epochs, feat_config)
# Take mean P300 amplitude across channels (first half = peak amp)
n_ch = p300.shape[1] // 2
p300_peak = p300[:, :n_ch].mean(axis=1)

p300_alc = p300_peak[alc_idx]
p300_ctrl = p300_peak[ctrl_idx]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([p300_alc, p300_ctrl], labels=['Alcoholic', 'Control'],
                patch_artist=True, widths=0.5)
bp['boxes'][0].set_facecolor('#E57373')
bp['boxes'][1].set_facecolor('#4FC3F7')
for box in bp['boxes']:
    box.set_edgecolor('#333')
    box.set_linewidth(2)
t_stat, p_val = stats.ttest_ind(p300_alc, p300_ctrl)
ax.set_ylabel('P300 Amplitude (μV)')
ax.set_title(f'P300: Alcoholic vs Control (t={t_stat:.2f}, p={p_val:.4f})')
plt.tight_layout()
plt.show()

print(f'Alcoholic P300: {p300_alc.mean():.4f} ± {p300_alc.std():.4f}')
print(f'Control P300: {p300_ctrl.mean():.4f} ± {p300_ctrl.std():.4f}')

## 4. Frontal Beta/Alpha Ratio

In [ ]:
ba_ratio = compute_frontal_beta_ratio(epochs, feat_config)
ba_alc = ba_ratio[alc_idx]
ba_ctrl = ba_ratio[ctrl_idx]

t_stat, p_val = stats.ttest_ind(ba_alc, ba_ctrl)
print(f'Frontal Beta/Alpha Ratio:')
print(f'  Alcoholic: {ba_alc.mean():.4f} ± {ba_alc.std():.4f}')
print(f'  Control: {ba_ctrl.mean():.4f} ± {ba_ctrl.std():.4f}')
print(f'  t={t_stat:.2f}, p={p_val:.4f}')